In [3]:
import os
import numpy as np

import tensorflow as tf
import tensorflow_hub as hub
import tensorflow_datasets as tfds

In [4]:
print("Version: ", tf.__version__)
print("Eager mode: ", tf.executing_eagerly())
print("Hub version: ", hub.__version__)
print("GPU is", "available" if tf.config.list_physical_devices("GPU") else "NOT AVAILABLE")

Version:  2.13.0
Eager mode:  True
Hub version:  0.14.0
GPU is NOT AVAILABLE


In [5]:
# Split the training set into 60% and 40% to end up with 15,000 examples
# for training, 10,000 examples for validation and 25,000 examples for testing.

train_data, validation_data, test_data = tfds.load(
    name="imdb_reviews",
    split=('train[:60%]', 'train[60%:]', 'test'),
    as_supervised=True)

Dl Completed...: 0 url [00:00, ? url/s]
Generating splits...:   0%|                                                                       | 0/3 [00:00<?, ? splits/s]
Generating train examples...: 0 examples [00:00, ? examples/s]
Generating train examples...: 1 examples [00:01,  1.84s/ examples]
Generating train examples...: 722 examples [00:01, 519.40 examples/s]
Generating train examples...: 1404 examples [00:02, 1106.75 examples/s]
Generating train examples...: 2156 examples [00:02, 1858.29 examples/s]
Generating train examples...: 2895 examples [00:02, 2652.72 examples/s]
Generating train examples...: 3626 examples [00:02, 3439.88 examples/s]
Generating train examples...: 4375 examples [00:02, 4222.87 examples/s]
Generating train examples...: 5103 examples [00:02, 4888.86 examples/s]
Generating train examples...: 5825 examples [00:02, 5441.75 examples/s]
Generating train examples...: 6564 examples [00:02, 5920.53 examples/s]
Generating train examples...: 7310 examples [00:02, 6323.0

Dataset imdb_reviews downloaded and prepared to C:\Users\Szesny\tensorflow_datasets\imdb_reviews\plain_text\1.0.0. Subsequent calls will reuse this data.


In [7]:
train_examples_batch, train_labels_batch = next(iter(train_data.batch(10)))
train_examples_batch                               

<tf.Tensor: shape=(10,), dtype=string, numpy=
array([b"This was an absolutely terrible movie. Don't be lured in by Christopher Walken or Michael Ironside. Both are great actors, but this must simply be their worst role in history. Even their great acting could not redeem this movie's ridiculous storyline. This movie is an early nineties US propaganda piece. The most pathetic scenes were those when the Columbian rebels were making their cases for revolutions. Maria Conchita Alonso appeared phony, and her pseudo-love affair with Walken was nothing but a pathetic emotional plug in a movie that was devoid of any real meaning. I am disappointed that there are movies like this, ruining actor's like Christopher Walken's good name. I could barely sit through it.",
       b'I have been known to fall asleep during films, but this is usually due to a combination of things including, really tired, being warm and comfortable on the sette and having just eaten a lot. However on this occasion I fell 

In [8]:
train_labels_batch

<tf.Tensor: shape=(10,), dtype=int64, numpy=array([0, 0, 0, 1, 1, 1, 0, 0, 0, 0], dtype=int64)>

In [19]:
embedding = "https://tfhub.dev/google/nnlm-en-dim128-with-normalization/2"
hub_layer = hub.KerasLayer(embedding, input_shape=[], dtype=tf.string, trainable=True)
hub_layer(train_examples_batch[:3])

<tf.Tensor: shape=(3, 128), dtype=float32, numpy=
array([[ 1.15015078e+00,  7.80129954e-02,  9.26615447e-02,
         2.83361465e-01,  9.67164431e-03, -1.49186030e-01,
         3.35665703e-01, -3.50244790e-01, -8.28830525e-03,
        -1.87713988e-02, -3.33069712e-02, -6.33094192e-01,
        -3.75421166e-01, -2.77732819e-01, -9.66175571e-02,
         1.72553658e-01, -1.33676559e-01,  3.80765833e-02,
        -2.75138170e-01,  4.94762301e-01,  3.93051691e-02,
         1.34496242e-01, -2.70728201e-01,  1.78942848e-02,
        -2.41071597e-01,  2.71089897e-02,  1.02333426e-01,
        -1.06628530e-01,  5.24298586e-02,  1.19170524e-01,
        -6.67077769e-03,  3.39231491e-01,  1.13014966e-01,
         1.06842607e-01,  3.91571254e-01, -1.89536318e-01,
        -1.74000308e-01, -1.06444173e-01, -1.34200469e-01,
         1.73583925e-01, -2.77695030e-01, -4.33591381e-02,
        -3.91500629e-02, -1.98340908e-01,  2.74854768e-02,
         2.76703000e-01,  1.40702859e-01, -3.14256102e-01,
      

In [20]:
model = tf.keras.Sequential()
model.add(hub_layer)
model.add(tf.keras.layers.Dense(16, activation='relu'))
model.add(tf.keras.layers.Dense(1))

model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 keras_layer_1 (KerasLayer)  (None, 128)               124642688 
                                                                 
 dense_2 (Dense)             (None, 16)                2064      
                                                                 
 dense_3 (Dense)             (None, 1)                 17        
                                                                 
Total params: 124644769 (475.48 MB)
Trainable params: 124644769 (475.48 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [21]:
model.compile(optimizer='adam',
              loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
              metrics=['accuracy'])

In [22]:
history = model.fit(train_data.shuffle(10000).batch(512), 
                    epochs=10,
                    validation_data=validation_data.batch(512),
                    verbose=1)

Epoch 1/10
30/30 [==============================] - 51s 2s/step - loss: 0.6041 - accuracy: 0.5810 - val_loss: 0.4997 - val_accuracy: 0.7456
Epoch 2/10
30/30 [==============================] - 50s 2s/step - loss: 0.3995 - accuracy: 0.8240 - val_loss: 0.3623 - val_accuracy: 0.8486
Epoch 3/10
30/30 [==============================] - 50s 2s/step - loss: 0.2593 - accuracy: 0.9069 - val_loss: 0.2973 - val_accuracy: 0.8754
Epoch 4/10
30/30 [==============================] - 50s 2s/step - loss: 0.1756 - accuracy: 0.9413 - val_loss: 0.2708 - val_accuracy: 0.8855
Epoch 5/10
30/30 [==============================] - 50s 2s/step - loss: 0.1179 - accuracy: 0.9644 - val_loss: 0.2642 - val_accuracy: 0.8896
Epoch 6/10
30/30 [==============================] - 50s 2s/step - loss: 0.0771 - accuracy: 0.9807 - val_loss: 0.2687 - val_accuracy: 0.8910
Epoch 7/10
30/30 [==============================] - 50s 2s/step - loss: 0.0487 - accuracy: 0.9903 - val_loss: 0.2772 - val_accuracy: 0.8904
Epoch 8/10
30/30 [==

In [23]:
results = model.evaluate(test_data.batch(512), verbose=2)

for name, value in zip(model.metrics_names, results):
    print("%s: %.3f" % (name, value))

49/49 - 7s - loss: 0.3583 - accuracy: 0.8694 - 7s/epoch - 141ms/step
loss: 0.358
accuracy: 0.869


In [28]:
examples = [
  "The movie was great!",
  "The movie was okay.",
  "The movie was terrible..."
]

export_model = tf.keras.Sequential([
  model,
  tf.keras.layers.Activation('sigmoid')
])

export_model.predict(examples)

1/1 [==============================] - 0s 174ms/step


array([[0.90739965],
       [0.0173384 ],
       [0.01747958]], dtype=float32)